# 03 - Baseline Regresi Logistik

Baseline memakai `build_features` dari `src/features.py`, jadi seluruh fitur
sudah diketahui saat pickup dan `volume_hub_hari` tidak dipakai. Evaluasi hanya
pada validation. Test April-Juni 2026 hanya dihitung jumlah barisnya.

In [1]:
from pathlib import Path
import json
import sys

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.features import FEATURE_COLUMNS_V1, build_features, make_logistic_preprocessor

data = pd.read_csv(ROOT / "data" / "clean.csv", parse_dates=["tanggal_pickup"])
train = data.loc[data["tanggal_pickup"] <= "2025-12-31"].copy()
valid = data.loc[data["tanggal_pickup"].between("2026-01-01", "2026-03-31")].copy()
test = data.loc[data["tanggal_pickup"].between("2026-04-01", "2026-06-30")].copy()
del data

ringkasan = pd.DataFrame(
    {
        "bagian": ["train", "validation", "test"],
        "baris": [len(train), len(valid), len(test)],
        "rasio_terlambat": [train["terlambat"].mean(), valid["terlambat"].mean(), None],
    }
)
ringkasan["rasio_terlambat"] = ringkasan["rasio_terlambat"].map(
    lambda x: "tidak dihitung" if pd.isna(x) else round(x, 4)
)
ringkasan

,bagian,baris,rasio_terlambat
0,train,26633,0.1778
1,validation,6623,0.2188
2,test,6744,tidak dihitung


## Fitur dan pipeline

`build_features` menghasilkan 18 fitur tanpa `volume_hub_hari`. Imputer, scaler,
dan one-hot encoder di-fit hanya pada train di dalam `Pipeline`.

In [2]:
X_train = build_features(train)
X_valid = build_features(valid)
y_train = train["terlambat"]
y_valid = valid["terlambat"]

model = Pipeline(
    [
        ("preprocessor", make_logistic_preprocessor()),
        ("logistic", LogisticRegression(max_iter=2000, random_state=42)),
    ]
)
model.fit(X_train, y_train)

proba_valid = model.predict_proba(X_valid)[:, 1]
pr_auc = average_precision_score(y_valid, proba_valid)
roc_auc = roc_auc_score(y_valid, proba_valid)
no_skill = y_valid.mean()

print("Fitur masukan:", len(FEATURE_COLUMNS_V1))
print(f"PR-AUC validation: {pr_auc:.4f}")
print(f"ROC-AUC validation: {roc_auc:.4f}")
print(f"PR-AUC no-skill: {no_skill:.4f}")

Fitur masukan: 18
PR-AUC validation: 0.4085
ROC-AUC validation: 0.6870
PR-AUC no-skill: 0.2188


## Simpan artefak baseline

Model yang disimpan sudah termasuk preprocessing. Metadata tidak memuat rasio
target test; test tetap tertutup sampai tahap akhir.

In [3]:
ARTIFACT_DIR = ROOT / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_VERSION = "engineered-logreg-v1"

metadata = {
    "model_version": MODEL_VERSION,
    "tipe_model": "LogisticRegression dengan build_features (tanpa volume_hub_hari)",
    "tanggal_latih": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
    "target": "terlambat",
    "feature_builder": "build_features_v1",
    "fitur_masukan": FEATURE_COLUMNS_V1,
    "fitur_dikeluarkan": ["volume_hub_hari", "flag_volume_ekstrem"],
    "alasan_fitur_dikeluarkan": "Total volume harian berpotensi baru lengkap setelah pickup.",
    "split": {
        "train": {"sampai": "2025-12-31", "baris": len(train), "rasio_terlambat": float(y_train.mean())},
        "validation": {"mulai": "2026-01-01", "sampai": "2026-03-31", "baris": len(valid), "rasio_terlambat": float(y_valid.mean())},
        "test": {"mulai": "2026-04-01", "sampai": "2026-06-30", "baris": len(test)},
    },
    "thresholds": {"kabari_pelanggan": 0.09, "prioritaskan": 0.30},
    "threshold_status": "placeholder; akan dikunci di notebook 05",
    "metrics": {
        "validation": {"pr_auc": float(pr_auc), "roc_auc": float(roc_auc), "pr_auc_no_skill": float(no_skill)},
        "test": None,
    },
    "library_versions": {
        "python": sys.version.split()[0],
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit-learn": sklearn.__version__,
        "joblib": joblib.__version__,
    },
    "test_disentuh": False,
}

joblib.dump(model, ARTIFACT_DIR / "model.joblib")
(ARTIFACT_DIR / "metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Model dan metadata baseline tersimpan:", MODEL_VERSION)

Model dan metadata baseline tersimpan: engineered-logreg-v1
